# UNSW-NB15 — C Tuning (Statevector / Ideal Regime)

## Mục đích
Chọn hyperparameter `C` (SVC) **fair giữa 4 kernel** trên UNSW-NB15:
`quantum` (FidelityStatevectorKernel, ZZFeatureMap reps=2, entanglement='full'), `linear`, `poly` (degree=2), `rbf`.

Kết quả cache vào `models_unsw/c_tuning_results.json` để các bước 1.4–1.5 dùng (không tune lại).

## Quy tắc khoa học
- **Tuning split**: chỉ `train_run1.parquet` — KHÔNG đụng vào run2..run5 để tránh data peek.
- **Zero-leakage**: pipeline `SelectKBest(K=35) → PCA(4) → MinMax[0,π]` được **fit lại trong từng CV fold**.
  Không tái sử dụng `pca_4components.joblib` pretrained (vì nó fit trên dữ liệu khác → leakage với tập tune).
- **CV**: StratifiedKFold(n_splits=5, shuffle=True, random_state=42).
- **Scoring**: `'f1'` (binary) — phù hợp với UNSW imbalanced (~74% Attack / 26% Normal).
- **Selection**: argmax theo CV mean F1.

## Output schema (`models_unsw/c_tuning_results.json`)
```json
{
  "quantum": {"C_best": ..., "cv_mean": ..., "cv_std": ..., "grid": [...], "scores_per_C": [{"C":..., "mean":..., "std":...}, ...]},
  "linear":  {...}, "poly": {...}, "rbf": {...},
  "metadata": {"n_folds": 5, "scoring": "f1", "tune_data": "train_run1", "n_samples": 100, "seed": 42, "date": "2026-05-15"}
}
```

In [1]:
import os, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms import QSVC

import qiskit, qiskit_machine_learning
print(f'Qiskit    : {qiskit.__version__}')
print(f'Qiskit ML : {qiskit_machine_learning.__version__}')

Qiskit    : 2.3.1
Qiskit ML : 0.9.0


In [2]:
# ── Cấu hình chung ─────────────────────────────────────────────────────
RANDOM_STATE = 42
N_QUBITS     = 4
ANGLE_MAX    = np.pi

# ZZFeatureMap (NISQ-compatible, khớp với toàn bộ pipeline QSVM-IDS)
REPS         = 2
ENTANGLEMENT = 'full'

# Pipeline dimensionality reduction (khớp với UNSW pretrained PCA n_features_in_=35)
K_SELECT     = 35
PCA_N        = 4
POLY_DEGREE  = 2
RBF_GAMMA    = 'scale'

# CV + grid
N_FOLDS      = 5
SCORING      = 'f1'
C_GRID       = [0.01, 0.1, 1.0, 10.0, 100.0]

# Paths
DATA_PATH       = '../data/unsw_nb15/processed_data/multi_run/train_run1.parquet'
MODELS_DIR      = '../models_unsw'
CACHE_DIR       = f'{MODELS_DIR}/c_tuning_cache'
FINAL_JSON_PATH = f'{MODELS_DIR}/c_tuning_results.json'
os.makedirs(CACHE_DIR, exist_ok=True)

LABEL_COLS = ['label', 'label_binary', 'label_multiclass', 'attack_category']

CONFIG_TAG = (
    f'r{REPS}_{ENTANGLEMENT}'
    f'_k{K_SELECT}_p{PCA_N}'
    f'_cv{N_FOLDS}_s{SCORING}'
    f'_run1'
)
CACHE_PATH = f'{CACHE_DIR}/c_tune_{CONFIG_TAG}.json'

np.random.seed(RANDOM_STATE)
print(f'CONFIG_TAG : {CONFIG_TAG}')
print(f'CACHE_PATH : {CACHE_PATH}')
print(f'FINAL JSON : {FINAL_JSON_PATH}')

CONFIG_TAG : r2_full_k35_p4_cv5_sf1_run1
CACHE_PATH : ../models_unsw/c_tuning_cache/c_tune_r2_full_k35_p4_cv5_sf1_run1.json
FINAL JSON : ../models_unsw/c_tuning_results.json


## Pipeline & chiến lược zero-leakage

Trong mỗi CV fold:
1. `SelectKBest(f_classif, K=35)` fit trên `X_raw[tr_idx], y[tr_idx]`
2. `PCA(n_components=4)` fit trên output bước 1 (chỉ trên train fold)
3. `MinMaxScaler(feature_range=(0, π))` fit trên output PCA (chỉ trên train fold)
4. Apply 3 bước transform lên `X[val_idx]` → `X_val_pi`
5. Huấn luyện SVC/QSVC trên `X_tr_pi`, đánh giá F1 trên `X_val_pi`

Quy trình này đảm bảo **không một mẫu validation nào** ảnh hưởng đến `SelectKBest.scores_`, `PCA.components_`, hoặc `MinMax.data_min_/data_max_`.

In [3]:
def fit_pipeline_on_train(X_tr_raw, y_tr):
    """Fit toàn bộ pipeline SelectKBest → PCA → MinMax[0,π] CHỈ trên train fold.

    Trả về (selector, pca, scaler) — dùng để transform val fold sau đó.
    """
    selector = SelectKBest(score_func=f_classif, k=K_SELECT).fit(X_tr_raw, y_tr)
    X_sel    = selector.transform(X_tr_raw)

    pca      = PCA(n_components=PCA_N, random_state=RANDOM_STATE).fit(X_sel)
    X_pca    = pca.transform(X_sel)

    scaler   = MinMaxScaler(feature_range=(0.0, ANGLE_MAX)).fit(X_pca)
    return selector, pca, scaler

def apply_pipeline(selector, pca, scaler, X_raw):
    # Áp dụng chuỗi transform theo đúng thứ tự đã fit trên train fold.
    return scaler.transform(pca.transform(selector.transform(X_raw)))

def build_quantum_kernel():
    """FidelityStatevectorKernel với ZZFeatureMap — deterministic/noiseless."""
    fm = zz_feature_map(
        feature_dimension=N_QUBITS,
        reps=REPS,
        entanglement=ENTANGLEMENT,
    )
    return FidelityStatevectorKernel(
        feature_map=fm,
        shots=None,
        enforce_psd=True,
        cache_size=None,
    )

In [4]:
# ── Load train_run1.parquet (canonical tuning split) ──────────────────
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'[MISSING] {DATA_PATH}')

df_run1 = pd.read_parquet(DATA_PATH)
feature_cols = [c for c in df_run1.columns if c not in LABEL_COLS]

X_raw = df_run1[feature_cols].to_numpy(dtype=np.float64)
y     = df_run1['label_binary'].to_numpy(dtype=np.int64)

print(f'Data path  : {DATA_PATH}')
print(f'X_raw      : {X_raw.shape} | features={len(feature_cols)}')
print(f'y dist     : {dict(zip(*np.unique(y, return_counts=True)))}')

Data path  : ../data/unsw_nb15/processed_data/multi_run/train_run1.parquet
X_raw      : (100, 186) | features=186
y dist     : {np.int64(0): np.int64(26), np.int64(1): np.int64(74)}


In [5]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  C TUNING — StratifiedKFold CV trên train_run1                       ║
# ║  Cache theo CONFIG_TAG → chạy lại không recompute                    ║
# ╚══════════════════════════════════════════════════════════════════════╝

def _tune_kernel(kernel_name, X_raw, y, c_grid):
    """Tune C cho 1 kernel — fit pipeline trong từng fold (zero-leakage).

    Trả về dict: {C_best, cv_mean, cv_std, grid, scores_per_C: [{C,mean,std}, ...]}
    """
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores_per_C = []

    for c in c_grid:
        fold_f1 = []
        for tr_idx, val_idx in skf.split(X_raw, y):
            # Fit pipeline CHỈ trên train fold
            sel, pca, sc = fit_pipeline_on_train(X_raw[tr_idx], y[tr_idx])
            X_tr_pi      = apply_pipeline(sel, pca, sc, X_raw[tr_idx])
            X_val_pi     = apply_pipeline(sel, pca, sc, X_raw[val_idx])

            # Huấn luyện classifier theo kernel
            if kernel_name == 'quantum':
                clf = QSVC(quantum_kernel=build_quantum_kernel(), C=c, random_state=RANDOM_STATE)
            elif kernel_name == 'linear':
                clf = SVC(kernel='linear', C=c, random_state=RANDOM_STATE)
            elif kernel_name == 'poly':
                clf = SVC(kernel='poly', degree=POLY_DEGREE, gamma='scale', C=c, random_state=RANDOM_STATE)
            elif kernel_name == 'rbf':
                clf = SVC(kernel='rbf', gamma=RBF_GAMMA, C=c, random_state=RANDOM_STATE)
            else:
                raise ValueError(f'Unknown kernel: {kernel_name}')

            clf.fit(X_tr_pi, y[tr_idx])
            fold_f1.append(f1_score(y[val_idx], clf.predict(X_val_pi), average='binary'))

        fold_f1 = np.asarray(fold_f1, dtype=float)
        scores_per_C.append({
            'C':    float(c),
            'mean': float(fold_f1.mean()),
            'std':  float(fold_f1.std(ddof=1)),
        })
        print(f'    C={c:<7}  cv_f1={fold_f1.mean():.4f} ± {fold_f1.std(ddof=1):.4f}')

    # Argmax theo mean F1
    best = max(scores_per_C, key=lambda r: r['mean'])
    return {
        'C_best':       best['C'],
        'cv_mean':      best['mean'],
        'cv_std':       best['std'],
        'grid':         list(c_grid),
        'scores_per_C': scores_per_C,
    }

# Load cache nếu có
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, encoding='utf-8') as fp:
        tune_results = json.load(fp)
    print(f'[CACHE] Đã load từ {CACHE_PATH}')
else:
    tune_results = {}
    t_total = time.time()
    for kname in ['linear', 'poly', 'rbf', 'quantum']:
        print(f'\n  [{kname}] tuning ...')
        t0 = time.time()
        tune_results[kname] = _tune_kernel(kname, X_raw, y, C_GRID)
        tune_results[kname]['elapsed_sec'] = float(time.time() - t0)
        print(f'    → C_best = {tune_results[kname]["C_best"]} | elapsed = {tune_results[kname]["elapsed_sec"]:.1f}s')
    print(f'\n[TOTAL] {time.time() - t_total:.1f}s')

    with open(CACHE_PATH, 'w', encoding='utf-8') as fp:
        json.dump(tune_results, fp, indent=2, ensure_ascii=False)
    print(f'[CACHE] Saved → {CACHE_PATH}')


  [linear] tuning ...
    C=0.01     cv_f1=0.8504 ± 0.0150
    C=0.1      cv_f1=0.8813 ± 0.0349
    C=1.0      cv_f1=0.8813 ± 0.0349
    C=10.0     cv_f1=0.8813 ± 0.0349
    C=100.0    cv_f1=0.8813 ± 0.0349
    → C_best = 0.1 | elapsed = 0.1s

  [poly] tuning ...
    C=0.01     cv_f1=0.8504 ± 0.0150
    C=0.1      cv_f1=0.8813 ± 0.0349
    C=1.0      cv_f1=0.8813 ± 0.0349
    C=10.0     cv_f1=0.8813 ± 0.0349
    C=100.0    cv_f1=0.8795 ± 0.0216
    → C_best = 0.1 | elapsed = 0.1s

  [rbf] tuning ...


    C=0.01     cv_f1=0.8504 ± 0.0150
    C=0.1      cv_f1=0.8504 ± 0.0150
    C=1.0      cv_f1=0.8813 ± 0.0349


    C=10.0     cv_f1=0.8813 ± 0.0349
    C=100.0    cv_f1=0.8729 ± 0.0446
    → C_best = 1.0 | elapsed = 0.1s

  [quantum] tuning ...


    C=0.01     cv_f1=0.8504 ± 0.0150


    C=0.1      cv_f1=0.8504 ± 0.0150


    C=1.0      cv_f1=0.8459 ± 0.0346


    C=10.0     cv_f1=0.8497 ± 0.0547


    C=100.0    cv_f1=0.7486 ± 0.0469
    → C_best = 0.01 | elapsed = 4.9s

[TOTAL] 5.1s
[CACHE] Saved → ../models_unsw/c_tuning_cache/c_tune_r2_full_k35_p4_cv5_sf1_run1.json


In [6]:
# ── Bảng kết quả + ghi JSON cuối cùng cho các bước 1.4–1.5 dùng ─────
import datetime

metadata = {
    'n_folds':    N_FOLDS,
    'scoring':    SCORING,
    'tune_data':  'train_run1',
    'n_samples':  int(X_raw.shape[0]),
    'n_features_raw': int(X_raw.shape[1]),
    'k_select':   K_SELECT,
    'pca_n':      PCA_N,
    'zz_reps':    REPS,
    'zz_entanglement': ENTANGLEMENT,
    'c_grid':     C_GRID,
    'selection_rule': 'argmax_cv_mean',
    'seed':       RANDOM_STATE,
    'date':       datetime.date.today().isoformat(),
    'config_tag': CONFIG_TAG,
}

final_payload = {k: tune_results[k] for k in ['quantum', 'linear', 'poly', 'rbf']}
final_payload['metadata'] = metadata

with open(FINAL_JSON_PATH, 'w', encoding='utf-8') as fp:
    json.dump(final_payload, fp, indent=2, ensure_ascii=False)
print(f'[OUTPUT] Saved → {FINAL_JSON_PATH}\n')

# Bảng tóm tắt
print(f'=== UNSW C TUNING SUMMARY ({N_FOLDS}-fold CV, scoring={SCORING}, n={X_raw.shape[0]}) ===')
print(f'  {"Kernel":>8}  {"C_best":>8}  {"CV mean":>8}  {"std":>7}  {"elapsed":>8}')
print('  ' + '-' * 50)
for k in ['quantum', 'linear', 'poly', 'rbf']:
    r = tune_results[k]
    elapsed = r.get('elapsed_sec', float('nan'))
    print(f'  {k:>8}  {r["C_best"]:>8.2f}  {r["cv_mean"]:>8.4f}  {r["cv_std"]:>7.4f}  {elapsed:>7.1f}s')

[OUTPUT] Saved → ../models_unsw/c_tuning_results.json

=== UNSW C TUNING SUMMARY (5-fold CV, scoring=f1, n=100) ===
    Kernel    C_best   CV mean      std   elapsed
  --------------------------------------------------
   quantum      0.01    0.8504   0.0150      4.9s
    linear      0.10    0.8813   0.0349      0.1s
      poly      0.10    0.8813   0.0349      0.1s
       rbf      1.00    0.8813   0.0349      0.1s
